In [51]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
#%matplotlib widget
%matplotlib inline
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch
import torch.optim as optim
import torch.nn as nn
from core.benchmarks import *
from core.CardiacCTdataset import DataLoaderFactory
from core.CNNmodel import *
from core.benchmarks import *
import pandas as pd
from core.model_utils import *
from core.CVsplits import *

import logging
from core.Log import *
import json
OUTER_FOLDS = 4; INNER_FOLDS = 3

logging.shutdown()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
setup_logger('OUTER')
# ['INNER_train', 'OUTER_train', 'OUTER_evaluate', 'Close']
logE = logging.getLogger('OUTER_evaluate')


In [ ]:
import itertools
import json

## 1. Define all model-specific hyperparameter sweeps in one dictionary
model_configs = {

#	"META+MLP": {
#		"LR_SWEEP": [2e-3, 5e-3],
#		"DR_SWEEP": [0.3],
#		"WD_SWEEP": [1e-3]
#	 ,
#	"RN18+MLP": {
#		"LR_SWEEP": [2e-4, 5e-4],
#		"DR_SWEEP": [0.2],
#		"WD_SWEEP": [1e-6]
#	},
#	"Axial":    {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4], "DR_SWEEP": [0.3]},
#	"Coronal":  {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4], "DR_SWEEP": [0.3]},
#	"Sagittal": {"LR_SWEEP": [2e-4, 5e-4], "WD_SWEEP": [1e-4], "DR_SWEEP": [0.3]},
}

# 2. Define global parameters that are the same for all models
GLOBAL_PARAMS = {
	"P": 4,
	"Epochs": 50,
}

CV_parameters = []
ID = 1
#( 7	, "0.0005"	, "1e-6" , "0.2")
parameter_sets = [
([1, 3], "8"	, "0.0005"	, "1e-4" , "0.2"),
([2], "16"	, "0.0008"	, "1e-4" , "0.3"),
([4], "7"	, "0.0005"	, "1e-6" , "0.2"),
#([1, 4], "15"	, "0.0008"	, "1e-6" , "0.3"),
#([3, 4], "14"	, "0.0008"	, "1e-4" , "0.2"),
#([2, 3], "11"	, "0.0005"	, "1e-6" , "0.4"),
]

CV_parameters = load_from_json("NCV_4_3_folds/OUTER_experiments.json")

for outer_fold_idx in range(1, 5):
	for (outer_folds, i, lr, wd, dr) in parameter_sets:
		if outer_fold_idx not in outer_folds:
			continue
		lr = float(lr)
		wd = float(wd)
		dr = float(dr)

		item = {
			"Model": "MultiViewCNN",
			'OUTER_FOLD': outer_fold_idx,
			'HPset': i,
			"LR": lr,
			"WD": wd,
			"DR": dr,
			"P": GLOBAL_PARAMS['P'],
			"Epochs": GLOBAL_PARAMS['Epochs'],
			"trained": False,
			"evaluated": False,
		}
		print(item)
		CV_parameters.append(item)
		ID += 1

print(f"Total combinations generated: {len(CV_parameters)}")

with open("NCV_4_3_folds/OUTER_experiments.json", "w") as f:
	json.dump(CV_parameters, f, indent=2)

CV_parameters


Loaded NCV_4_3_folds/OUTER_experiments.json.
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 1, 'HPset': '8', 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.2, 'P': 4, 'Epochs': 50, 'trained': False, 'evaluated': False}
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 2, 'HPset': '16', 'LR': 0.0008, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 50, 'trained': False, 'evaluated': False}
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 3, 'HPset': '8', 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.2, 'P': 4, 'Epochs': 50, 'trained': False, 'evaluated': False}
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 4, 'HPset': '7', 'LR': 0.0005, 'WD': 1e-06, 'DR': 0.2, 'P': 4, 'Epochs': 50, 'trained': False, 'evaluated': False}
Total combinations generated: 13


[{'Model': 'MultiViewCNN',
  'OUTER_FOLD': 1,
  'HPset': '15',
  'LR': 0.0008,
  'WD': 1e-06,
  'DR': 0.3,
  'P': 4,
  'Epochs': 50,
  'trained': False,
  'evaluated': False},
 {'Model': 'MultiViewCNN',
  'OUTER_FOLD': 2,
  'HPset': '11',
  'LR': 0.0005,
  'WD': 1e-06,
  'DR': 0.4,
  'P': 4,
  'Epochs': 50,
  'trained': False,
  'evaluated': False},
 {'Model': 'MultiViewCNN',
  'OUTER_FOLD': 2,
  'HPset': '8',
  'LR': 0.0005,
  'WD': 0.0001,
  'DR': 0.2,
  'P': 4,
  'Epochs': 50,
  'trained': False,
  'evaluated': False},
 {'Model': 'MultiViewCNN',
  'OUTER_FOLD': 2,
  'HPset': '9',
  'LR': 0.0005,
  'WD': 1e-06,
  'DR': 0.3,
  'P': 4,
  'Epochs': 50,
  'trained': False,
  'evaluated': False},
 {'Model': 'MultiViewCNN',
  'OUTER_FOLD': 3,
  'HPset': '14',
  'LR': 0.0008,
  'WD': 0.0001,
  'DR': 0.2,
  'P': 4,
  'Epochs': 50,
  'trained': False,
  'evaluated': False},
 {'Model': 'MultiViewCNN',
  'OUTER_FOLD': 3,
  'HPset': '11',
  'LR': 0.0005,
  'WD': 1e-06,
  'DR': 0.4,
  'P': 4,
  '

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, brier_score_loss
import numpy as np
import copy

logT = logging.getLogger('OUTER_train')
logE = logging.getLogger('OUTER_evaluate')


def train_OUTER_model(model, train_loader, val_loader, experiment):
	log = logging.getLogger('OUTER_train')
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	#feature_extractor, model =create_adapted_resnet18(device)
	model.to(device)
	epochs = experiment['Epochs']
	model_name  = experiment['Model']
	out=experiment['OUTER_FOLD']
	HP = experiment['HPset']
	LR = experiment['LR']
	WD = experiment['WD']
	DR = experiment['DR']
	P = experiment['P']
	TH = 0.5
	rel_thresh = 5e-3 if np.isclose(LR, 5e-4) else 3e-3
	sch_patience, sch_cooldown = 2, 1

	ES_PATIENCE = max(P, sch_patience + sch_cooldown + 2)
	alpha_ema = 0.30  # smoothing for EMA of val loss

	optimizer = optim.Adam(model.parameters(), lr= LR, weight_decay=WD)
	scheduler = ReduceLROnPlateau(optimizer, mode='min',
								  patience=sch_patience, factor=0.5,
								  threshold=rel_thresh, threshold_mode='rel',
								  cooldown=sch_cooldown, min_lr=1e-6)
	criterion = nn.BCEWithLogitsLoss()


	# --- best trackers ---
	best_val_loss = np.inf
	best_epoch    = -1
	best_lr_at_best = LR
	auc_at_best   = np.nan
	accuracy_at_best = np.nan
	best_model_state = None
	brier_at_best = np.nan

	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)

	# --- EMA & patience ---
	ema_val = None
	best_ema = np.inf
	no_improve = 0

	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)

	print(f"	train_N: {train_N}, val_N: {val_N} ↳ Training {model_name} | Training model... ")
	for epoch in range(epochs):
		model.train()
		running_loss = 0.0
		y_true, y_pred = [], []
		for batch in train_loader:
			axi = batch["axial_image"].to(device)
			cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)

			optimizer.zero_grad(set_to_none=True)

			logits = model(axi, cor, sag, met)
			#logits = model(axi, met)
			#logits = model(cor, met)
			#logits = model(sag, met)
			T_loss = criterion(logits, lbl)

			T_loss.backward()
			optimizer.step()

			running_loss += T_loss.item() * lbl.size(0)
			with torch.no_grad():
				probs = torch.sigmoid(logits)
				preds = (probs > TH).long()
				y_true.append(lbl.detach().cpu().numpy())
				y_pred.append(preds.detach().cpu().numpy())

		TrainLoss = running_loss / max(1, train_N)
		y_true = np.concatenate(y_true).reshape(-1)
		y_pred = np.concatenate(y_pred).reshape(-1)
		TrainAcc  = (y_true == y_pred).mean()

		model.eval()
		running_loss = 0.0
		y_true, y_prob, y_pred = [], [], []

		with torch.no_grad():
			for batch in val_loader:
				axi = batch["axial_image"].to(device)
				cor = batch["coronal_image"].to(device)
				sag = batch["sagittal_image"].to(device)
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)

				logits = model(axi, cor, sag, met)

				#logits = model(axi, met)
				#logits = model(cor, met)
				#logits = model(sag, met)
				V_loss = criterion(logits, lbl)
				running_loss += V_loss.item() * lbl.size(0)

				probs = torch.sigmoid(logits)
				preds = (probs > TH).long()

				y_true.append(lbl.detach().cpu().numpy())
				y_prob.append(probs.detach().cpu().numpy())
				y_pred.append(preds.detach().cpu().numpy())

		ValLoss = running_loss / max(1, val_N)

		y_true  = np.concatenate(y_true).reshape(-1)
		y_prob  = np.concatenate(y_prob).reshape(-1)
		y_pred  = np.concatenate(y_pred).reshape(-1)

		ValAcc  = (y_true == y_pred).mean()
		AUC = roc_auc_score(y_true, y_prob)
		Brier = brier_score_loss(y_true, y_prob)

		# -------------- EMA + scheduler --------------
		ema_val = ValLoss if ema_val is None else alpha_ema*ValLoss + (1 - alpha_ema)*ema_val
		prev_lr = optimizer.param_groups[0]['lr']
		scheduler.step(ema_val)  # schedule on EMA, not raw ValLoss
		new_lr = optimizer.param_groups[0]['lr']
		lr_drop = int(new_lr < prev_lr)



		# -------------- early stopping test -----------
		improved = ema_val < best_ema * (1 - rel_thresh)
		if improved:
			best_ema = ema_val
			best_val_loss = ValLoss
			best_epoch = epoch
			best_lr_at_best = new_lr
			accuracy_at_best = ValAcc
			auc_at_best = AUC
			no_improve = 0
			best_model_state = copy.deepcopy(model.state_dict())
		else:
			no_improve += 1
		es_triggered = int(no_improve >= ES_PATIENCE)
		line=f"{model_name};{out};{HP};{epoch};{TrainLoss};{TrainAcc};{ValLoss};{ValAcc};{AUC};{Brier};{ema_val};{new_lr};{no_improve};{lr_drop};{es_triggered};{best_val_loss};{best_epoch}"
		log.info(line)
		if es_triggered: break

	# ----- final return (best state + summary for outer fold) -----
	summary = {
		"Model": model_name,
		"OuterFold": out,
		"DR": DR,
		"LR": LR,
		"WD": WD,
		"BestEpoch": best_epoch,
		"BestValLoss": float(best_val_loss),
		"AUC_at_Best": float(auc_at_best) if auc_at_best is not None else np.nan,
		"Accuracy_at_Best": float(accuracy_at_best),
		"Brier_at_Best": float(brier_at_best),
		"LR_at_Best": float(best_lr_at_best),
		"ES_Patience_Used": ES_PATIENCE,
		"RelThresh": rel_thresh,
		"EMA_alpha": alpha_ema,
	}
	return summary, best_model_state



def append_experiment_results(item, path="NCV_4_3_folds/OUTER_summary.jsonl"):
	with open(path, "a") as f:  # append mode
		f.write(json.dumps(item) + "\n")



In [7]:
main_dataset = load_dataset_info(file="data/data_info.json")
DL = DataLoaderFactory(main_dataset)


In [32]:
CV_parameters = load_from_json("NCV_4_3_folds/OUTER_experiments.json")
df = pd.DataFrame(CV_parameters)
df


Loaded NCV_4_3_folds/OUTER_experiments.json.


,Model,OUTER_FOLD,HPset,LR,WD,DR,P,Epochs,trained,evaluated
0,MultiViewCNN,1,15,0.0008,0.000001,0.3,4,50,True,True
1,MultiViewCNN,2,11,0.0005,0.000001,0.4,4,50,True,True
2,MultiViewCNN,2,8,0.0005,0.000100,0.2,4,50,True,True
3,MultiViewCNN,2,9,0.0005,0.000001,0.3,4,50,True,True
4,MultiViewCNN,3,14,0.0008,0.000100,0.2,4,50,True,True
5,MultiViewCNN,3,11,0.0005,0.000001,0.4,4,50,True,True
6,MultiViewCNN,4,15,0.0008,0.000001,0.3,4,50,True,True
7,MultiViewCNN,4,14,0.0008,0.000100,0.2,4,50,True,True
8,MultiViewCNN,4,16,0.0008,0.000100,0.3,4,50,True,True
9,MultiViewCNN,1,8,0.0005,0.000100,0.2,4,50,False,False


In [4]:
#cid, y, yh, p, z in zip(case_ids_all, y_true, y_hat, y_prob, y_logit)
logE.info(f"model_name;out;HP;cid;y_true;y_hat;y_prob;y_logit;TH;LR;WD;DR ")


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from core.CVsplits import *
import logging
import json

def append_experiment_results(item, path="NCV/single_inner_summary.jsonl"):
	with open(path, "a") as f:  # append mode
		f.write(json.dumps(item) + "\n")

def EVALUATE_MODEL(model, test_loader, experiment):
	log = logging.getLogger('OUTER_evaluate')
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	#feature_extractor, model =create_adapted_resnet18(device)
	model.to(device)
	model_name  = experiment['Model']
	out = experiment['OUTER_FOLD']
	DR = experiment['DR']
	HP = experiment['HPset']
	LR = experiment['LR']
	WD = experiment['WD']
	TH = 0.5

	model.eval()
	running_loss = 0.0
	y_true, y_prob, y_logit = [], [], []
	case_ids_all = []
	criterion = nn.BCEWithLogitsLoss()
	eval_N = len(test_loader.dataset)

	#with torch.no_grad():
	with torch.inference_mode():
		print(f"	↳ Experiment | Evaluating {model_name} on fold {out}... ")
		for batch in test_loader:
			axi = batch["axial_image"].to(device)
			cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1).float()
			cids = batch["CaseID"]

			#axi_features = feature_extractor(axi)
			#cor_features = feature_extractor(cor)
			#sag_features = feature_extractor(sag)

			#combined_input = torch.cat([axi_features, cor_features, sag_features, met], dim=1)

			#logits = model(combined_input)
			#logits = model(sag, meta=met)
			#logits = model(cor, meta=met)
			#logits = model(axi, meta=met)
			logits = model(axi, cor, sag, met)
			#logits = model(meta=met)

			loss = criterion(logits, lbl)
			running_loss += loss.item() * lbl.size(0)


			probs = torch.sigmoid(logits).squeeze(1)
			y_true.append(lbl.squeeze(1).cpu().numpy())
			y_prob.append(probs.cpu().numpy())
			y_logit.append(logits.squeeze(1).cpu().numpy())
			case_ids_all.extend(cids)

	y_true  = np.concatenate(y_true).astype(int)
	y_prob  = np.concatenate(y_prob).astype(float)
	y_logit = np.concatenate(y_logit).astype(float)
	y_hat   = (y_prob > TH).astype(int)
	final_loss = running_loss / eval_N
	results = {"Model": model_name,
		"OuterFold": out,
		"HPset": HP,
		"DR": DR,
		"LR": LR,
		"WD": WD,
		"Loss": final_loss,
		"case_ids": case_ids_all,
		"y_true": y_true.tolist(),
		"y_prob": y_prob.tolist(),
		"y_logit": y_logit.tolist(),
		"y_hat": y_hat.tolist(),
		"EvalN": int(eval_N)
		}

	for cid, y, yh, p, z in zip(case_ids_all, y_true, y_hat, y_prob, y_logit):
		log.info(f"{model_name};{out};{HP};{cid};{int(y)};{int(yh)};{float(p)};{float(z)};{TH};{LR};{WD};{DR}")
	append_experiment_results(results, path="NCV/OUTER_evaluation.jsonl")


def num_trainable_params(m):
	return sum(p.numel() for p in m.parameters() if p.requires_grad)



In [ ]:
CV_parameters = load_from_json("NCV_4_3_folds/OUTER_experiments.json")
def is_completed(exp):
	return (
		exp.get("Model") == "MultiViewCNN"
		and exp.get("OUTER_FOLD") == 4
		#and exp.get("INNER_FOLD") == 3
		#and exp.get("HPset") not in ["8"]
		and exp.get("HPset") not in ["14", "15", "16"]
		#and exp.get("HPset") in {7, 8, 9, 10, 11, 12, 13, 14, 15, 16}
	)

updated = 0
for exp in CV_parameters:
	if is_completed(exp):
		exp["trained"] = True          # normalize to lowercase
		exp["evaluated"] = False          # normalize to lowercase
		updated += 1

print(f"Marked {updated} experiments as trained=True.")

with open("NCV_4_3_folds/OUTER_experiments.json", "w") as f:
	json.dump(CV_parameters, f, indent=2)
print("OUTER_experiments.json updated.")


Loaded NCV_4_3_folds/OUTER_experiments.json.
Marked 1 experiments as trained=True.
OUTER_experiments.json updated.


In [38]:
CV_parameters = load_from_json("NCV_4_3_folds/OUTER_experiments.json")
filtered = [
	exp for exp in CV_parameters
	if exp["Model"] ==  "MultiViewCNN"      # MultiViewCNN, Coronal, Sagittal, Axial, META+MLP, RN18+MLP
	and exp["OUTER_FOLD"] == 3       #2, 3, 4
	and exp["HPset"] in ["11", "14"]
	#and exp["HPset"] in [1, 3, 4]
	#and exp["ExpID"] == 186
	#and exp["trained"] == False
	#and exp["evaluated"] == False
]
print(f"experiments to do: {len(filtered)}")   #30
for experiment in filtered:
	#print(experiment)
	HPset = experiment['HPset']
	OUT = experiment['OUTER_FOLD']
	#INN = experiment['INNER_FOLD']
	print(f"out: {OUT}, hpset: {HPset}")


Loaded NCV_4_3_folds/OUTER_experiments.json.
experiments to do: 2
out: 3, hpset: 14
out: 3, hpset: 11


In [ ]:

for experiment in filtered:
	print(experiment)
	HPset = experiment['HPset']
	OUT = experiment['OUTER_FOLD']
	model_type = experiment["Model"]
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	_, _, test_loader = DL.create_outer_loaders(OUT-1)
	DR = experiment['DR']
	#model = SingleViewClassifier(DR)
	model = MultiViewCNN(DR)
	model.to(device) # Add this line to move the model to the GPU
	state_dict = torch.load(f"pth_models/{model_type}_F_{OUT}_{HPset}_final.pth", map_location=torch.device('cpu'))
	model.load_state_dict(state_dict)
	model_sum = num_trainable_params(model)
	EVALUATE_MODEL(model, test_loader, experiment)
	#break








{'Model': 'MultiViewCNN', 'OUTER_FOLD': 1, 'HPset': '8', 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.2, 'P': 4, 'Epochs': 50, 'trained': False, 'evaluated': False}
	↳ Experiment | Evaluating MultiViewCNN on fold 1... 
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 2, 'HPset': '16', 'LR': 0.0008, 'WD': 0.0001, 'DR': 0.3, 'P': 4, 'Epochs': 50, 'trained': False, 'evaluated': False}
	↳ Experiment | Evaluating MultiViewCNN on fold 2... 
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 3, 'HPset': '8', 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.2, 'P': 4, 'Epochs': 50, 'trained': False, 'evaluated': False}
	↳ Experiment | Evaluating MultiViewCNN on fold 3... 
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 4, 'HPset': '7', 'LR': 0.0005, 'WD': 1e-06, 'DR': 0.2, 'P': 4, 'Epochs': 50, 'trained': True, 'evaluated': False}
	↳ Experiment | Evaluating MultiViewCNN on fold 4... 


In [ ]:
import torch, torch.nn.functional as F
import numpy as np



CV_parameters = load_from_json("NCV_4_3_folds/OUTER_experiments.json")
filtered = [
	exp for exp in CV_parameters
	if exp["Model"] ==  "MultiViewCNN"      # MultiViewCNN, Coronal, Sagittal, Axial, META+MLP, RN18+MLP
	and exp["OUTER_FOLD"] == 3       #2, 3, 4
	and exp["HPset"] in ["11", "14"]
]
print(f"experiments to do: {len(filtered)}")
for experiment in filtered:
	#print(experiment)
	HPset = experiment['HPset']
	OUT = experiment['OUTER_FOLD']
	print(f"out: {OUT}, hpset: {HPset}")


Loaded NCV_4_3_folds/OUTER_experiments.json.
experiments to do: 2
out: 3, hpset: 14
out: 3, hpset: 11


In [52]:

for experiment in filtered:
	print(experiment)
	HPset = experiment['HPset']
	OUT = experiment['OUTER_FOLD']
	model_type = experiment["Model"]
	_, _, test_loader = DL.create_outer_loaders(OUT-1)
	break


model.load_state_dict(torch.load(f"pth_models/{model_type}_F_{OUT}_{HPset}_final.pth",  map_location=torch.device('cpu')))
model.to("cuda" if torch.cuda.is_available() else "cpu")

# Point to the **last conv** (or whichever) in each branch:
targets = {
	"axial":    model.axial_branch.bn3,      # <- adapt names
	"coronal":  model.coronal_branch.bn3,
	"sagittal": model.sagittal_branch.bn3,
}

res = gradcam_eval(
	model, test_loader, target_layers=targets,
	TH=0.5, case_ids_filter=None, target="pred", max_cases=30,
	save_npz_path=f"cams_npz/gradcam_F{OUT}_HP{HPset}_pred.npz",
)

#append_experiment_results(res, path="NCV/gradcam_evaluation.jsonl")
res


{'Model': 'MultiViewCNN', 'OUTER_FOLD': 3, 'HPset': '14', 'LR': 0.0008, 'WD': 0.0001, 'DR': 0.2, 'P': 4, 'Epochs': 50, 'trained': True, 'evaluated': True}


{'case_ids': ['TTS_GB_10468072_81F',
  'CNTRL_LLM_24775207_66F',
  'TTS_MB_12229480_96F',
  'TTS_MGR_2545663_93M',
  'TTS_EIC_26844753_77F',
  'TTS_APH_3516218_67F',
  'CNTRL_GKW_27839323_65M',
  'CNTRL_DL_232051_72F',
  'TTS_DJD_6122410_88F',
  'TTS_PMS_6674980_78F',
  'CNTRL_SBMD_50080962_45F',
  'CNTRL_HTNA_39573258_80F',
  'CNTRL_PLB_32190472_65F',
  'TTS_JOL_25671702_69F',
  'CNTRL_DMJ_07659402_76F',
  'CNTRL_BWB_08372559_71M',
  'CNTRL_DB_24229841_67F',
  'CNTRL_DVMcC_39192026_54M',
  'TTS_BEH_4037842_77F',
  'CNTRL_SBAE_22469423_79F',
  'TTS_FFJ_11086170_86F',
  'CNTRL_HB_50159284_62F',
  'CNTRL_SDA_50042444_68M',
  'TTS_LAM_5515176_65F',
  'TTS_BFM_10624765_79F',
  'TTS_WF_10497121_62F',
  'CNTRL_AG_519880_37F',
  'TTS_JMMcC_4147591_77F',
  'TTS_MG_22519714_73M',
  'TTS_AD_12191953_51F'],
 'y_true': array([1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0,
        0, 1, 1, 1, 0, 1, 1, 1]),
 'y_prob': array([0.5412448 , 0.49981058, 0.70719928, 0.16924168, 0.559886

In [48]:
loaded_data = np.load(f"cams_npz/gradcam_F{OUT}_HP{HPset}_pred.npz", allow_pickle=True)
results = {k: v.tolist() if k != 'cams' else None for k, v in loaded_data.items()}
results['cams'] = {
    "axial": list(loaded_data['cams_axial']),
    "coronal": list(loaded_data['cams_coronal']),
    "sagittal": list(loaded_data['cams_sagittal']),
}

print("ax bn3:", results["axial"].shape)      # expect [B, 64, H/8, W/8] if 3×2×2 pools
print("sag bn3:", results["sagittal"].shape)
print("cor bn3:", results["coronal"].shape)

# Make sure case_ids is a list for the .index() method to work
results['case_ids'] = list(loaded_data['case_ids'])


# 3. Now you can safely call your plotting function
cid = results["case_ids"][0]
show_case_cams(results, test_loader, case_id=cid)
plt.show()


KeyError: 'axial'